In [0]:
data = spark.sql("""select * from sales_project_db.bronze_sales""")

In [0]:
from pyspark.sql.functions import col,round,when
from pyspark.sql.functions import datediff
from pyspark.sql.functions import year, month, dayofmonth, weekofyear

df_cleaned= (
    data
# Remove null values
    .dropna()
    .dropDuplicates()
    .orderBy("Order_Date")
    .dropDuplicates(["Order_ID","Product_ID"])
    .drop("Product_Name")
    .withColumn("Total_sales", round((data["Quantity"]*data["Sales"]),4))
    .withColumnRenamed("Profit","Residue")
    .withColumnRenamed("Sub-Category","Sub_Category")
    .withColumn("Profit_Loss", when(col("Residue")>0, "P").otherwise("L"))
    .withColumn("delivery_days",datediff(col("Ship_Date"), col("Order_Date")))
    .withColumn("delivery_type",when(col("delivery_days") == 1, "1-day")
                   .when(col("delivery_days") <= 3, "fast")
                   .otherwise("delayed"))
    .withColumn("order_year", year("Order_Date")) 
       .withColumn("order_month", month("Order_Date")) 
       .withColumn("order_day", dayofmonth("Order_Date")) 
       .withColumn("order_week", weekofyear("Order_Date"))
    .withColumn("profit_margin", col("Residue") / col("Sales"))

)

In [0]:
df_cleaned.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("sales_project_db.silver_sales")